# 01 — Embedding Visualization

Map and visualize the hBehaveMAE latent space using **temporally independent** chunks,
colored by **strain**, **age**, and **experimental stage**.

### Recording structure
Each mouse has **6 recordings**: 3 experimental stages (HAB, ACQ, …) × 2 ages (Adult, Old).
Video name: `HDP-013893_ACQ_Old_4` → `animal_id / exp_stage / age / session`.

### Independence
3000-frame chunks, 1800-frame gaps (> 900 receptive field → zero leakage).

### Filtering
1. Merge with metadata → 2. Drop `B6CAST-129SPWK-F2` → 3. Keep strains with ≥ 20 chunks.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import umap

sns.set_context("notebook", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
print("Libraries loaded.")

## 1. Configuration

In [ ]:
from pathlib import Path

def find_base_dir(anchor="data"):
    """Walk upward from CWD until we find a child directory named `anchor`."""
    cwd = Path.cwd()
    for parent in [cwd] + list(cwd.parents):
        if (parent / anchor).is_dir():
            return parent
    raise FileNotFoundError(
        f"Could not find a '{anchor}/' directory in any parent of {cwd}."
    )

BASE_DIR = find_base_dir("data")
PROJECT_ROOT = Path("/scratch/michal/projects/dvc_ofd_2025")
EMB_H5_PATH = BASE_DIR / "data/embeddings/ofd_tailhip_20260226-205043.h5"
META_TSV_PATH = PROJECT_ROOT / "data/raw/openfield_ORT/hdp_meta.tsv"
# --- Independence parameters ---
# hBehaveMAE receptive field = 900 frames.
# 3000-frame chunks with 1800-frame gaps → zero information leakage.
STAGE = "stage2"            # 192D
CHUNK_FRAMES = 3000
GAP_FRAMES = 1800
STRIDE = CHUNK_FRAMES + GAP_FRAMES
# --- Filtering ---
HYBRID_STRAIN = "B6CAST-129SPWK-F2"
MIN_CHUNKS_PER_STRAIN = 20

UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.3
UMAP_METRIC = "euclidean"
RANDOM_STATE = 42

print(f"BASE_DIR: {BASE_DIR}")
print(f"Embedding: {EMB_H5_PATH}  (exists: {EMB_H5_PATH.exists()})")
print(f"Metadata:  {META_TSV_PATH}")

## 2. Load Metadata

In [ ]:
meta_df = pd.read_csv(META_TSV_PATH, sep=r'\s+')
meta_df.columns = meta_df.columns.str.replace('"', '').str.strip()
meta_df['animal_id'] = meta_df['animal_id'].astype(str).str.replace('"', '').str.strip()

strain_lookup = dict(zip(meta_df['animal_id'], meta_df['strain']))
print(f"Metadata loaded: {len(meta_df)} animals, {meta_df['strain'].nunique()} unique strains")

## 3. Extract Independent Chunks

In [ ]:
def extract_independent_chunks(embeddings, chunk_frames=3000, gap_frames=1800):
    """Slice (T, D) into independent chunks separated by gaps.

    If the video is shorter than chunk_frames, the entire video is
    returned as a single chunk (graceful fallback so no video is
    silently dropped when chunk_frames is set very large).
    """
    T = embeddings.shape[0]

    # Fallback: video shorter than one chunk → use the whole video
    if T < chunk_frames:
        return [embeddings]

    stride = chunk_frames + gap_frames
    chunks = []
    start = 0
    while start + chunk_frames <= T:
        chunks.append(embeddings[start : start + chunk_frames])
        start += stride
    return chunks
all_chunks = []
chunk_meta_rows = []

with h5py.File(EMB_H5_PATH, 'r') as f:
    video_names = sorted(f.keys())
    print(f"Videos in H5 file: {len(video_names)}")

    for vid in video_names:
        if STAGE not in f[vid]:
            continue

        emb = f[vid][STAGE][:]         # (T, 192)

        # Parse: HDP-013893_ACQ_Old_4 → animal_id, exp_stage, age
        parts = vid.split('_')
        if len(parts) < 4:
            continue
        animal_id = parts[0]
        exp_stage = parts[1]           # HAB, ACQ, etc.
        age       = parts[2]           # Adult, Old

        chunks = extract_independent_chunks(emb, CHUNK_FRAMES, GAP_FRAMES)

        for ci, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            chunk_meta_rows.append({
                "pos": len(all_chunks) - 1,
                "video_name": vid,
                "animal_id": animal_id,
                "exp_stage": exp_stage,
                "age": age,
                "chunk_idx": ci,
            })

chunk_df = pd.DataFrame(chunk_meta_rows)
print(f"\nExtracted {len(all_chunks)} independent chunks")
print(f"  Videos:  {chunk_df['video_name'].nunique()}")
print(f"  Animals: {chunk_df['animal_id'].nunique()}")
print(f"  Exp stages: {sorted(chunk_df['exp_stage'].unique())}")
print(f"  Ages:       {sorted(chunk_df['age'].unique())}")
print(f"\nRecordings per animal (videos): "
      f"{chunk_df.groupby('animal_id')['video_name'].nunique().describe()[['mean','min','max']].to_dict()}")

## 4. Data Filtering & Mean Pooling

In [ ]:
print("=" * 60)
print("Applying Strict Data Filtering")
print("=" * 60)

# Step 1: Merge with metadata
master_df = pd.merge(chunk_df, meta_df, on='animal_id', how='inner')
print(f"After metadata merge: {len(master_df)} chunks")

if master_df.empty:
    raise RuntimeError(
        f"Merge produced 0 rows! Check animal_id formats.\n"
        f"  chunk IDs: {chunk_df['animal_id'].unique()[:3]}\n"
        f"  meta IDs:  {meta_df['animal_id'].unique()[:3]}"
    )

# Step 2: Drop missing strain
master_df = master_df.dropna(subset=['strain'])
print(f"After dropping missing strain: {len(master_df)}")

# Step 3: Remove hybrid strain
n_hybrid = (master_df['strain'] == HYBRID_STRAIN).sum()
master_df = master_df[master_df['strain'] != HYBRID_STRAIN]
print(f"Removed {n_hybrid} chunks from '{HYBRID_STRAIN}': {len(master_df)} remain")

# Step 4: Minimum chunk count per strain
strain_counts = master_df['strain'].value_counts()
valid_strains = strain_counts[strain_counts >= MIN_CHUNKS_PER_STRAIN].index
n_dropped = master_df['strain'].nunique() - len(valid_strains)
master_df = master_df[master_df['strain'].isin(valid_strains)]
print(f"Removed {n_dropped} rare strains (< {MIN_CHUNKS_PER_STRAIN} chunks): {len(master_df)} remain")

# Step 5: Sync numpy arrays
filtered_chunks = [all_chunks[i] for i in master_df['pos'].values]
master_df = master_df.reset_index(drop=True)

print(f"\nFinal dataset: {len(master_df)} chunks, "
      f"{master_df['strain'].nunique()} strains, "
      f"{master_df['animal_id'].nunique()} animals")
print(f"\nBreakdown by exp_stage × age:")
print(master_df.groupby(['age', 'exp_stage']).size().unstack(fill_value=0))


# Mean-pool each chunk → single 192D fingerprint
chunk_means = np.array([c.mean(axis=0) for c in filtered_chunks])
print(f"\nPooled matrix: {chunk_means.shape}")

## 5. PCA

In [ ]:
scaler = StandardScaler()
chunk_means_scaled = scaler.fit_transform(chunk_means)

n_pca = min(50, *chunk_means_scaled.shape)
pca = PCA(n_components=n_pca, random_state=RANDOM_STATE)
pca_result = pca.fit_transform(chunk_means_scaled)
cumvar = np.cumsum(pca.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, n_pca+1), pca.explained_variance_ratio_, color="steelblue", alpha=0.7)
axes[0].set_xlabel("PC"); axes[0].set_ylabel("Variance Ratio"); axes[0].set_title("Scree Plot")
axes[0].set_xlim(0.5, 20.5)

axes[1].plot(range(1, n_pca+1), cumvar, 'o-', markersize=3, color="steelblue")
axes[1].axhline(0.90, color='red', ls='--', alpha=0.6, label='90%')
axes[1].axhline(0.95, color='green', ls='--', alpha=0.6, label='95%')
axes[1].set_xlabel("# Components"); axes[1].set_ylabel("Cumulative Variance"); axes[1].legend()
plt.tight_layout(); plt.show()

for t in [0.80, 0.90, 0.95]:
    print(f"  {t:.0%} variance → {np.searchsorted(cumvar, t)+1} PCs")

## 6. Scatter Plots — Colored by Strain Family

With ~90 strains, per-strain legends are unreadable.
We group strains into **genetic families** for the legend, and provide a
secondary plot with a continuous colormap for individual strain detail.

In [ ]:
def get_strain_family(strain_name):
    """Map a strain to its broad genetic family for readable plotting."""
    s = str(strain_name)
    if s.startswith("BXD"):     return "BXD (recombinant inbred)"
    if s.startswith("CC"):      return "CC (collaborative cross)"
    if "C57" in s:              return "C57 family"
    if "C3H" in s:              return "C3H family"
    if s.startswith("DBA"):     return "DBA family"
    if s in ("CAST/EiJ", "LEWES/EiJ", "PWK/PhJ", "WSB/EiJ",
             "CAROLI/EiJ", "PANCEVO/EiJ", "SKIVE/EiJ", "PERC/EiJ",
             "MSM/MsJ", "JF1/MsJ", "PWD/PhJ"):
                                return "Wild-derived"
    return "Other inbred"

master_df["strain_family"] = master_df["strain"].map(get_strain_family)
families = sorted(master_df["strain_family"].unique())
family_palette = dict(zip(families, sns.color_palette("Set2", len(families))))

# Also build numeric codes for individual-strain colormap plots
unique_strains = sorted(master_df['strain'].unique())
strain_codes = pd.Categorical(master_df['strain'], categories=unique_strains).codes

# Strains to annotate on the colormap plots
ANNOTATE_STRAINS = ["CAST/EiJ", "PWK/PhJ", "WSB/EiJ", "C57BL/6J", "DBA/2J", "NZW/LacJ"]

def scatter_by_family(ax, embedding, title):
    for family in families:
        mask = (master_df["strain_family"] == family).values
        ax.scatter(embedding[mask, 0], embedding[mask, 1],
                   label=f"{family}  (n={mask.sum()})",
                   s=15, alpha=0.5, color=family_palette[family], edgecolors='none')
    ax.set_title(title)
    ax.legend(markerscale=3, fontsize=8, framealpha=0.9)

def scatter_individual(ax, embedding, title):
    ax.scatter(embedding[:, 0], embedding[:, 1],
               c=strain_codes, cmap='nipy_spectral', s=10, alpha=0.5, edgecolors='none')
    ax.set_title(title)
    for s in ANNOTATE_STRAINS:
        mask = (master_df['strain'] == s).values
        if mask.any():
            cx, cy = embedding[mask, 0].mean(), embedding[mask, 1].mean()
            ax.annotate(s, (cx, cy), fontsize=7, fontweight='bold', ha='center', va='bottom',
                        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7, lw=0.5))

# --- PCA ---
fig, axes = plt.subplots(1, 2, figsize=(22, 8))
scatter_by_family(axes[0], pca_result[:,:2],
    f"PCA — Strain Families\nPC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}")
scatter_individual(axes[1], pca_result[:,:2],
    f"PCA — All {len(unique_strains)} Strains (colormap)")
plt.tight_layout(); plt.show()

## 7. UMAP

In [ ]:
print("Fitting UMAP...")
reducer = umap.UMAP(n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST,
                     metric=UMAP_METRIC, n_components=2, random_state=RANDOM_STATE)
umap_result = reducer.fit_transform(chunk_means_scaled)
print(f"UMAP done: {umap_result.shape}")

fig, axes = plt.subplots(1, 2, figsize=(22, 8))
scatter_by_family(axes[0], umap_result, "UMAP — Strain Families")
scatter_individual(axes[1], umap_result, f"UMAP — All {len(unique_strains)} Strains (colormap)")
plt.tight_layout(); plt.show()

## 8. UMAP Colored by Age & Experimental Stage

These are the key biological variables beyond strain.
If the embeddings capture behavioral differences due to aging or task phase,
we should see separation here.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 8))

# --- Age ---
ax = axes[0]
for age in sorted(master_df['age'].unique()):
    mask = (master_df['age'] == age).values
    ax.scatter(umap_result[mask, 0], umap_result[mask, 1],
               label=f"{age}  (n={mask.sum()})", s=12, alpha=0.4, edgecolors='none')
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.set_title("UMAP — Colored by Age")
ax.legend(markerscale=3, fontsize=10)

# --- Experimental Stage ---
ax = axes[1]
for stage in sorted(master_df['exp_stage'].unique()):
    mask = (master_df['exp_stage'] == stage).values
    ax.scatter(umap_result[mask, 0], umap_result[mask, 1],
               label=f"{stage}  (n={mask.sum()})", s=12, alpha=0.4, edgecolors='none')
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.set_title("UMAP — Colored by Experimental Stage")
ax.legend(markerscale=3, fontsize=10)

plt.tight_layout(); plt.show()

## 9. PCA Colored by Age & Experimental Stage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 8))

ax = axes[0]
for age in sorted(master_df['age'].unique()):
    mask = (master_df['age'] == age).values
    ax.scatter(pca_result[mask, 0], pca_result[mask, 1],
               label=f"{age}  (n={mask.sum()})", s=12, alpha=0.4, edgecolors='none')
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("PCA — Colored by Age"); ax.legend(markerscale=3)

ax = axes[1]
for stage in sorted(master_df['exp_stage'].unique()):
    mask = (master_df['exp_stage'] == stage).values
    ax.scatter(pca_result[mask, 0], pca_result[mask, 1],
               label=f"{stage}  (n={mask.sum()})", s=12, alpha=0.4, edgecolors='none')
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("PCA — Colored by Exp Stage"); ax.legend(markerscale=3)

plt.tight_layout(); plt.show()

## 10. Intra-Animal vs. Inter-Animal Variance

Highlight a few animals with many chunks.  If same-animal chunks cluster tightly,
the embeddings capture individual identity beyond strain.

In [ ]:
chunks_per_animal = master_df.groupby('animal_id').size().sort_values(ascending=False)
top_animals = chunks_per_animal.head(6).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
for ax, emb, title in zip(axes, [pca_result[:,:2], umap_result], ["PCA", "UMAP"]):
    ax.scatter(emb[:, 0], emb[:, 1], s=5, c='lightgray', alpha=0.3, label='_nolegend_')
    for animal_id, color in zip(top_animals, sns.color_palette("bright", len(top_animals))):
        mask = (master_df['animal_id'] == animal_id).values
        strain = master_df.loc[mask, 'strain'].iloc[0]
        ax.scatter(emb[mask, 0], emb[mask, 1], s=40, color=color, alpha=0.8,
                   edgecolors='k', linewidth=0.3, label=f"{animal_id} ({strain})")
    ax.set_title(f"{title} — Top 6 Animals Highlighted")
    ax.legend(fontsize=7, markerscale=1.5)
plt.tight_layout(); plt.show()

## 11. Quantifying Intra- vs. Inter-Animal Distances

In [ ]:
from sklearn.metrics.pairwise import cosine_distances

results_rows = []
for strain in unique_strains:
    strain_mask = (master_df['strain'] == strain).values
    animals = master_df.loc[strain_mask, 'animal_id'].unique()
    if len(animals) < 2: continue

    intra, inter = [], []
    for aid in animals:
        amask = (master_df['animal_id'] == aid).values
        aembs = chunk_means_scaled[amask]
        if len(aembs) >= 2:
            d = cosine_distances(aembs)
            intra.extend(d[np.triu_indices(len(aembs), k=1)].tolist())

    for i, a1 in enumerate(animals):
        for a2 in animals[i+1:]:
            e1 = chunk_means_scaled[(master_df['animal_id'] == a1).values]
            e2 = chunk_means_scaled[(master_df['animal_id'] == a2).values]
            inter.extend(cosine_distances(e1, e2).ravel().tolist())

    if intra and inter:
        results_rows.append({"strain": strain, "n_animals": len(animals),
                             "intra_animal": np.mean(intra), "inter_animal": np.mean(inter)})

dist_df = pd.DataFrame(results_rows)
dist_df["ratio"] = dist_df["inter_animal"] / dist_df["intra_animal"]

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(dist_df))
ax.bar(x - 0.175, dist_df['intra_animal'], 0.35, label='Intra-Animal', color='steelblue')
ax.bar(x + 0.175, dist_df['inter_animal'], 0.35, label='Inter-Animal', color='coral')
ax.set_xticks(x); ax.set_xticklabels(dist_df['strain'], rotation=90, fontsize=6)
ax.set_ylabel("Mean Cosine Distance"); ax.set_title("Intra vs Inter-Animal Distance"); ax.legend()
plt.tight_layout(); plt.show()

## 12. UMAP by Video (batch effects)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
video_codes = pd.Categorical(master_df['video_name']).codes
ax.scatter(umap_result[:, 0], umap_result[:, 1], c=video_codes, cmap='tab20', s=8, alpha=0.5)
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.set_title("UMAP Colored by Video — Batch Effect Check")
plt.tight_layout(); plt.show()
print("Tight per-video clusters → possible recording confounds.")

## 13. Linear Discriminant Analysis (LDA) — Label-Aware Visualization

Unlike PCA and UMAP (unsupervised), **LDA** is a **supervised projection** that uses
class labels to find directions maximizing between-class separation relative to
within-class spread:

$$\frac{\text{between-class variance}}{\text{within-class variance}}$$

This makes LDA a diagnostic for whether the hBehaveMAE embeddings encode
**biologically meaningful structure** — without training a classifier.

| Method | Uses labels | Goal |
|--------|------------|------|
| PCA | No | maximize variance |
| UMAP | No | preserve manifold |
| **LDA** | **Yes** | **maximize class separation** |

With *C* classes, LDA produces at most *C − 1* projection axes.
We project onto the first two for visualization.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# We'll project the same scaled, mean-pooled chunk embeddings used everywhere else.
# LDA needs more samples than features per class, and n_components <= n_classes - 1.

fig, axes = plt.subplots(1, 3, figsize=(26, 7))

targets = [
    ("Strain",           master_df['strain'].values,    unique_strains),
    ("Age",              master_df['age'].values,        sorted(master_df['age'].unique())),
    ("Exp Stage",        master_df['exp_stage'].values,  sorted(master_df['exp_stage'].unique())),
]

for ax, (label, y_vals, classes) in zip(axes, targets):
    n_classes = len(classes)
    n_components = min(n_classes - 1, chunk_means_scaled.shape[1], 2)

    if n_components < 1:
        ax.set_title(f"LDA — {label}\n(need ≥ 2 classes)")
        continue

    lda = LinearDiscriminantAnalysis(n_components=n_components)
    lda_result = lda.fit_transform(chunk_means_scaled, y_vals)

    # Explained variance ratio of the discriminant axes
    ev = lda.explained_variance_ratio_

    if n_components == 1:
        # Only 1 axis (e.g. Age with 2 classes) — plot as 1D with jitter
        for cls in classes:
            mask = (y_vals == cls)
            jitter = np.random.RandomState(42).normal(0, 0.15, size=mask.sum())
            ax.scatter(lda_result[mask, 0], jitter, label=cls, s=12, alpha=0.5, edgecolors='none')
        ax.set_xlabel(f"LD1 ({ev[0]:.1%} of discriminant variance)")
        ax.set_ylabel("jitter")
    else:
        # 2 axes — standard 2D scatter
        if label == "Strain":
            # Too many strains for a legend — use strain families
            for family in families:
                mask = (master_df["strain_family"] == family).values
                ax.scatter(lda_result[mask, 0], lda_result[mask, 1],
                           label=f"{family}  (n={mask.sum()})",
                           s=12, alpha=0.4, color=family_palette[family], edgecolors='none')
        else:
            for cls in classes:
                mask = (y_vals == cls)
                ax.scatter(lda_result[mask, 0], lda_result[mask, 1],
                           label=cls, s=12, alpha=0.4, edgecolors='none')
        ax.set_xlabel(f"LD1 ({ev[0]:.1%})")
        ax.set_ylabel(f"LD2 ({ev[1]:.1%})" if len(ev) > 1 else "")

    ax.set_title(f"LDA — {label}  ({n_classes} classes, {n_components} axes)")
    ax.legend(markerscale=2.5, fontsize=7, framealpha=0.9)

plt.suptitle("Linear Discriminant Analysis — Label-Aware Projections", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Print explained variance
for label, y_vals, classes in targets:
    n_c = min(len(classes) - 1, chunk_means_scaled.shape[1])
    if n_c < 1: continue
    lda_tmp = LinearDiscriminantAnalysis(n_components=min(n_c, 10))
    lda_tmp.fit(chunk_means_scaled, y_vals)
    ev = lda_tmp.explained_variance_ratio_
    print(f"\n{label} ({len(classes)} classes):")
    for i, v in enumerate(ev[:5]):
        print(f"  LD{i+1}: {v:.1%}")
    if len(ev) > 5:
        print(f"  ... ({len(ev)} total axes)")

## 14. LDA Deep Dive — Strain (Individual Strain Colormap)

The strain LDA above uses family colors for readability. Here we show the same
projection with individual-strain coloring and centroid annotations.

In [ ]:
n_strain_axes = min(len(unique_strains) - 1, chunk_means_scaled.shape[1], 2)
lda_strain = LinearDiscriminantAnalysis(n_components=n_strain_axes)
lda_strain_result = lda_strain.fit_transform(chunk_means_scaled, master_df['strain'].values)
ev_s = lda_strain.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(22, 8))

# Left: strain family colors
ax = axes[0]
for family in families:
    mask = (master_df["strain_family"] == family).values
    ax.scatter(lda_strain_result[mask, 0], lda_strain_result[mask, 1],
               label=f"{family}", s=12, alpha=0.4,
               color=family_palette[family], edgecolors='none')
ax.set_xlabel(f"LD1 ({ev_s[0]:.1%})"); ax.set_ylabel(f"LD2 ({ev_s[1]:.1%})")
ax.set_title("LDA (Strain) — Strain Families")
ax.legend(markerscale=3, fontsize=8)

# Right: individual strain colormap + centroid annotations
ax = axes[1]
ax.scatter(lda_strain_result[:, 0], lda_strain_result[:, 1],
           c=strain_codes, cmap='nipy_spectral', s=10, alpha=0.4, edgecolors='none')
ax.set_xlabel(f"LD1 ({ev_s[0]:.1%})"); ax.set_ylabel(f"LD2 ({ev_s[1]:.1%})")
ax.set_title(f"LDA (Strain) — All {len(unique_strains)} Strains")

for s in ANNOTATE_STRAINS:
    mask = (master_df['strain'] == s).values
    if mask.any():
        cx, cy = lda_strain_result[mask, 0].mean(), lda_strain_result[mask, 1].mean()
        ax.annotate(s, (cx, cy), fontsize=7, fontweight='bold', ha='center', va='bottom',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7, lw=0.5))

plt.tight_layout(); plt.show()